In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [2]:
df = pd.read_csv(
    "../dataset/fraudTrain.csv"
)


df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [3]:
df.drop(
    columns=[
        "Unnamed: 0",
        "trans_num",
        "first",
        "last",
        "street",
        "dob"
    ],
    inplace=True,
    errors="ignore"
)

In [4]:
df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"]
)


df["hour"] = df["trans_date_trans_time"].dt.hour

df["day"] = df["trans_date_trans_time"].dt.day


df.drop(
    "trans_date_trans_time",
    axis=1,
    inplace=True
)

In [5]:
encoder = LabelEncoder()


for col in df.select_dtypes(include="object").columns:

    df[col] = encoder.fit_transform(df[col])

In [6]:
X = df.drop(
    "is_fraud",
    axis=1
)


y = df["is_fraud"]



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [7]:
smote = SMOTE(
    random_state=42
)


X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [ ]:

import mlflow

print(mlflow.__version__)


mlflow.set_tracking_uri(
    "http://127.0.0.1:5000"
)


mlflow.set_experiment(
    "Anomaly Detection MLflow"
)


print("MLflow Connected")

2026/06/16 20:23:23 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection MLflow' does not exist. Creating a new experiment.


2.10.2
MLflow Connected


In [22]:
from sklearn.metrics import (
    classification_report,
    accuracy_score
)


def log_metrics(y_true, y_pred):

    report = classification_report(
        y_true,
        y_pred,
        output_dict=True
    )


    mlflow.log_metric(
        "accuracy",
        accuracy_score(
            y_true,
            y_pred
        )
    )


    mlflow.log_metric(
        "recall_class_0",
        report["0"]["recall"]
    )


    mlflow.log_metric(
        "recall_class_1",
        report["1"]["recall"]
    )


    mlflow.log_metric(
        "f1_score_macro",
        report["macro avg"]["f1-score"]
    )

In [26]:
with mlflow.start_run(
    run_name="Logistic Regression"
):

    model = LogisticRegression(
        C=1,
        solver="liblinear",
        random_state=42
    )


    model.fit(
        X_train,
        y_train
    )


    predictions = model.predict(
        X_test
    )


    mlflow.log_param(
        "C",
        1
    )


    mlflow.log_param(
        "solver",
        "liblinear"
    )


    log_metrics(
        y_test,
        predictions
    )


    mlflow.sklearn.log_model(
        model,
        "model"
    )

c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to co

In [ ]:

with mlflow.start_run(
    run_name="Random Forest"
):

    model = RandomForestClassifier(
        n_estimators=30,
        max_depth=3,
        random_state=42
    )


    model.fit(
        X_train,
        y_train
    )


    pred = model.predict(
        X_test
    )


    mlflow.log_param(
        "n_estimators",
        30
    )


    mlflow.log_param(
        "max_depth",
        3
    )


    log_metrics(
        y_test,
        pred
    )


    mlflow.sklearn.log_model(
        model,
        "model"
    )

c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [ ]:

with mlflow.start_run(
    run_name="XGBoost"
):

    model = XGBClassifier(
        eval_metric="logloss",
        random_state=42
    )


    model.fit(
        X_train,
        y_train
    )


    pred = model.predict(
        X_test
    )


    mlflow.log_param(
        "eval_metric",
        "logloss"
    )


    log_metrics(
        y_test,
        pred
    )


    mlflow.xgboost.log_model(
        model,
        "model"
    )

c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\sklearn.py:1116: UserWarning: [20:45:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\_distutils_ha

In [ ]:

with mlflow.start_run(
    run_name="XGBoost + SMOTE"
):

    model = XGBClassifier(
        eval_metric="logloss",
        random_state=42
    )


    model.fit(
        X_train_smote,
        y_train_smote
    )


    pred = model.predict(
        X_test
    )


    mlflow.log_param(
        "eval_metric",
        "logloss"
    )


    mlflow.log_param(
        "sampling",
        "SMOTE"
    )


    log_metrics(
        y_test,
        pred
    )


    mlflow.xgboost.log_model(
        model,
        "model"
    )

c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\sklearn.py:1116: UserWarning: [20:46:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\_distutils_ha

In [30]:
from mlflow.tracking import MlflowClient

In [31]:
client = MlflowClient()


experiment = client.get_experiment_by_name(
    "Anomaly Detection MLflow"
)


runs_df = mlflow.search_runs(
    experiment_ids=[
        experiment.experiment_id
    ]
)


runs_df = runs_df.sort_values(
    by=[
        "metrics.recall_class_1",
        "metrics.f1_score_macro"
    ],
    ascending=False
)


runs_df[
    [
        "run_id",
        "tags.mlflow.runName",
        "metrics.accuracy",
        "metrics.recall_class_1",
        "metrics.f1_score_macro"
    ]
]

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.recall_class_1,metrics.f1_score_macro
0,9f64314fc21142969af1ff95a7a958f3,XGBoost + SMOTE,0.987221,0.936945,0.726340
1,d4d9f92f4ef54e76a427411f20b7bb4e,XGBoost,0.998499,0.810835,0.930690
2,56b373965d484a3db5cf165971176e98,Random Forest,0.994221,0.001776,0.500324
3,c3a3cf2feafe48498420bf2898c79087,Logistic Regression,0.994211,0.000000,0.498549


In [44]:
best_run = runs_df.iloc[0]


best_run_id = best_run["run_id"]


best_model_name = best_run["tags.mlflow.runName"]


print(best_run_id)
print(best_model_name)

9f64314fc21142969af1ff95a7a958f3
XGBoost + SMOTE


In [46]:
from mlflow.tracking import MlflowClient

client = MlflowClient()


client.set_registered_model_alias(
    name="anomaly-detection-prod",
    alias="challenger",
    version="1"
)


client.set_registered_model_alias(
    name="anomaly-detection-prod",
    alias="champion",
    version="1"
)

In [47]:
model = mlflow.pyfunc.load_model(
    "models:/anomaly-detection-prod/1"
)


predictions = model.predict(
    X_test
)


predictions[:10]

c:\Users\Prathmesh Gonjare\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\sklearn.py:1125: UserWarning: [21:17:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1509: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)


array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0])